<a href="https://colab.research.google.com/github/Leopqs/Treinamento_de_IAs/blob/main/SegundoTrabalho.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Dados de exemplo com base nas imagens fornecidas
data = {
    'Idade': ['infantil', 'infantil', 'infantil', 'infantil', 'adolescente', 'adolescente', 'adolescente', 'adolescente', 'adolescente', 'adolescente', 'adolescente', 'adulto', 'adulto', 'adulto', 'adulto'],
    'Diagnóstico': ['miopia', 'miopia', 'hipermetropia', 'hipermetropia', 'miopia', 'miopia', 'miopia', 'hipermetropia', 'hipermetropia', 'hipermetropia', 'hipermetropia', 'miopia', 'miopia', 'hipermetropia', 'hipermetropia'],
    'Astigmatismo': ['não', 'sim', 'não', 'sim', 'não', 'sim', 'não', 'não', 'sim', 'sim', 'não', 'não', 'sim', 'não', 'sim'],
    'Taxa lacrimal': ['reduzida', 'normal', 'normal', 'normal', 'reduzida', 'reduzida', 'normal', 'reduzida', 'normal', 'normal', 'normal', 'normal', 'normal', 'normal', 'normal'],
    'Tipo de lente': ['nenhuma', 'gelatinosa', 'gelatinosa', 'dura', 'gelatinosa', 'gelatinosa', 'nenhuma', 'gelatinosa', 'dura', 'gelatinosa', 'gelatinosa', 'gelatinosa', 'gelatinosa', 'gelatinosa', 'gelatinosa']
}

# Criar um DataFrame com os dados
df = pd.DataFrame(data)

# Suavização Laplaciana (α = 1)
alpha = 1

# Função para calcular a probabilidade de uma classe com Naive Bayes
def naive_bayes(df, patient):
    # Contagens dos valores por classe
    class_counts = df['Tipo de lente'].value_counts()
    total_records = len(df)

    # Probabilidade a priori de cada classe
    class_prob = class_counts / total_records

    # Calcular as probabilidades condicionais com suavização laplaciana
    conditional_prob = {}
    for column in df.columns[:-1]:  # Excluir a última coluna (Tipo de lente)
        for value in df[column].unique():
            for label in class_counts.index:
                # Subconjunto dos dados que pertencem à classe
                subset = df[df['Tipo de lente'] == label]
                # Contagem de cada valor de atributo condicionado à classe
                count = subset[column].value_counts().get(value, 0)
                # Suavização Laplaciana
                prob = (count + alpha) / (len(subset) + alpha * len(df[column].unique()))
                conditional_prob[(column, value, label)] = prob

    # Calcular as probabilidades do paciente para cada classe
    patient_prob = {}
    for label in class_counts.index:
        prob = class_prob[label]
        for column, value in patient.items():
            prob *= conditional_prob.get((column, value, label), 0)
        patient_prob[label] = prob

    # Normalizar as probabilidades
    total_prob = sum(patient_prob.values())
    normalized_prob = {key: value / total_prob for key, value in patient_prob.items()}

    return normalized_prob

# Exemplo de paciente
patient = {
    'Idade': 'infantil',
    'Diagnóstico': 'hipermetropia',
    'Astigmatismo': 'não',
    'Taxa lacrimal': 'reduzida'
}

# Resultado das probabilidades normalizadas
probabilidades = naive_bayes(df, patient)

# Exibir as probabilidades
for classe, prob in probabilidades.items():
    print(f'Probabilidade de {classe}: {prob:.4f}')

Probabilidade de gelatinosa: 0.6515
Probabilidade de nenhuma: 0.2324
Probabilidade de dura: 0.1162
